In [ ]:
%pip install elasticsearch==8.19.0
%restart_python

In [ ]:
"""
Sync Vector Index — Qwen3 (works-vectors-v2)   [oxjob #1275]

Fork of sync_vector_index.ipynb for the multilingual re-embed. Differences:
  * source table  openalex.vector_search.work_embeddings_qwen3 (473.6M Qwen3-Embedding-0.6B vectors)
  * target index  works-vectors-v2, created here with element_type=bfloat16 (ES 9.3+, halves the
                  raw-vector copy: ~1.5 TB instead of ~2.36 TB) and 8 primary shards, 0 replicas
  * its own staging/checkpoint tables (_qwen3) so a v1 sync can still run alongside
The v1 notebook is untouched and keeps serving prod until the flip.

PREREQ: the vector deployment needs ~1.5 TB free ON TOP of works-vectors-v1 (2.06 TB) — add a third
data node before a full sync, and remove it after v1 is dropped.

Two-phase semantic search: kNN here returns IDs → mget full docs from the works index (the `works` alias).

Run modes:
- is_full_sync=true: Load all ~413M embeddings (initial load, ~6-8 hours)
- is_full_sync=false: the staging table the nightly `stage` task built (embedded in the last 2 days + works whose
  filter fields changed, oxjob #1433), or rows embedded in the last `lookback_days` (default 7) if it is missing. Used by the nightly job
  (jobs/embed_qwen3_nightly.yaml) right after EmbedQwen3Incremental; index ops overwrite, so re-sending is safe.
"""

import json
from datetime import datetime
from dataclasses import dataclass

from pyspark.sql import functions as F
from pyspark.sql.types import *
from elasticsearch import Elasticsearch, helpers
import logging

logging.basicConfig(level=logging.WARNING, format='[%(asctime)s]: %(message)s')
log = logging.getLogger(__name__)

ELASTIC_INDEX = "works-vectors-v2"
ELASTIC_URL = dbutils.secrets.get(scope="elastic", key="es_vector_search_url")

IS_FULL_SYNC = dbutils.widgets.get("is_full_sync").lower() == "true"
try:
    LOOKBACK_DAYS = int(dbutils.widgets.get("lookback_days"))
except Exception:  # widget absent (e.g. the full-sync job) → default
    LOOKBACK_DAYS = 7



def _widget(name, default):
    """Widget value, or `default` when the widget is absent or blank (oxjob #1433)."""
    try:
        value = dbutils.widgets.get(name).strip()
    except Exception:
        return default
    return value or default


# oxjob #1433: a one-off run (e.g. a filter-field refresh) passes its own staging/checkpoint tables so it can't collide
# with the nightly's; resume=true keeps an incremental run's checkpoint instead of truncating it.
STAGING_TABLE = _widget("staging_table", "openalex.vector_search.vector_sync_staging_qwen3")
CHECKPOINT_TABLE = _widget("checkpoint_table", "openalex.vector_search.vector_sync_checkpoint_qwen3")
NUM_BATCHES = int(_widget("num_batches", "200" if IS_FULL_SYNC else "10"))
RESUME = _widget("resume", "false").lower() == "true"
# oxjob #1433: the index keeps its own copy of 14 filter fields. The stage task lists works that no longer exist
# (DELETES_TABLE) and stages each row with the hash of its filter fields; after a clean sync those hashes go into
# MIRROR_TABLE, which the next stage task diffs against. "none" switches either off.
DELETES_TABLE = _widget("deletes_table", "openalex.vector_search.vector_sync_deletes_qwen3")
MAX_DELETES = int(_widget("max_deletes", "1000000"))
MIRROR_TABLE = _widget("mirror_table", "openalex.vector_search.vector_filter_fields_sent")
CLEANUP = _widget("cleanup", "true").lower() == "true"
# oxjob #1433: pacing for big in-place re-sends. The vector cluster (2 nodes, 1.7 TB index, no replica) serves kNN from
# the page cache; the merges that tens of millions of overwrites set off evict it, and semantic search failed 60-78% of
# requests on 29 Sep. MAX_BATCHES caps the batches sent per run (0 = all); MAX_MINUTES stops starting batches after that
# long. With KNN_GUARD_MS > 0, before each batch the run waits until the index has at most MAX_MERGES merges running and
# a direct kNN probe takes at most KNN_GUARD_MS; if that doesn't happen within GUARD_WAIT_MINUTES it stops for the
# night. A run that leaves batches undone skips the deletes, the mirror and the cleanup; resume=true picks it up.
MAX_BATCHES = int(_widget("max_batches", "0"))
MAX_MINUTES = int(_widget("max_minutes", "0"))
KNN_GUARD_MS = int(_widget("knn_guard_ms", "0"))
MAX_MERGES = int(_widget("max_merges", "4"))
GUARD_WAIT_MINUTES = int(_widget("guard_wait_minutes", "30"))

print(f"IS_FULL_SYNC: {IS_FULL_SYNC}  LOOKBACK_DAYS (incremental only): {LOOKBACK_DAYS}")
print(f"Staging: {STAGING_TABLE}  checkpoint: {CHECKPOINT_TABLE}  batches: {NUM_BATCHES}  resume: {RESUME}")
print(f"Deletes: {DELETES_TABLE} (max {MAX_DELETES:,})  mirror: {MIRROR_TABLE}  cleanup: {CLEANUP}")
print(f"Pacing: max_batches {MAX_BATCHES or 'all'}  max_minutes {MAX_MINUTES or 'none'}  "
      f"knn_guard_ms {KNN_GUARD_MS or 'off'}  max_merges {MAX_MERGES}  guard_wait_minutes {GUARD_WAIT_MINUTES}")
print(f"Target index: {ELASTIC_INDEX}")

In [ ]:
# Create works-vectors-v2 with the Qwen3 mapping if it does not exist (oxjob #1275).
# dims/similarity match v1; element_type bfloat16 is the only mapping change (validated on the
# live 9.3.2 cluster 2026-09-21). int8_hnsw quantization as before.
_client = Elasticsearch(hosts=[ELASTIC_URL], request_timeout=180, max_retries=5, retry_on_timeout=True)
try:
    if not _client.indices.exists(index=ELASTIC_INDEX):
        _client.indices.create(index=ELASTIC_INDEX, body={
            "settings": {"number_of_shards": 8, "number_of_replicas": 0, "refresh_interval": "-1"},
            "mappings": {"properties": {
                "id": {"type": "keyword"},
                "vector_embedding": {"type": "dense_vector", "dims": 1024, "element_type": "bfloat16",
                                     "index": True, "similarity": "cosine",
                                     "index_options": {"type": "int8_hnsw", "m": 16, "ef_construction": 100}},
                "publication_year": {"type": "integer"},
                "type": {"type": "keyword"},
                "is_oa": {"type": "boolean"},
                "language": {"type": "keyword"},
                "author_ids": {"type": "keyword"},
                "institution_ids": {"type": "keyword"},
                "country_codes": {"type": "keyword"},
                "is_retracted": {"type": "boolean"},
                "source_id": {"type": "keyword"},
                "cited_by_count": {"type": "integer"},
                "funder_ids": {"type": "keyword"},
                "has_fulltext": {"type": "boolean"},
                "has_abstract": {"type": "boolean"},
                "license_id": {"type": "keyword"},
            }}})
        print(f"Created {ELASTIC_INDEX}")
    else:
        print(f"{ELASTIC_INDEX} already exists")
finally:
    _client.close()


In [ ]:
# Set replicas to 0 for faster bulk indexing during full sync
if IS_FULL_SYNC:
    try:
        client = Elasticsearch(
            hosts=[ELASTIC_URL],
            request_timeout=180,
            max_retries=5,
            retry_on_timeout=True
        )
        if client.indices.exists(index=ELASTIC_INDEX):
            client.indices.put_settings(index=ELASTIC_INDEX, body={
                "index": {
                    "number_of_replicas": 0,
                    "refresh_interval": "-1"
                }
            })
            print(f"Set replicas=0, refresh=-1 on {ELASTIC_INDEX} for full sync")
        else:
            print(f"Index {ELASTIC_INDEX} does not exist")
    finally:
        client.close()

### Prepare Data

Join work_embeddings_v2 with openalex_works to get the 14 filter fields.
Flatten authorships into arrays: author_ids, institution_ids, country_codes, funder_ids.

In [ ]:
import time

# Batch processing configuration: STAGING_TABLE, CHECKPOINT_TABLE, NUM_BATCHES come from the widgets (cell 1)

# --- Checkpoint table ---
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {CHECKPOINT_TABLE} (
        batch_id INT, indexed_count LONG, skipped_count LONG,
        error_count LONG, completed_at TIMESTAMP
    )
""")
completed_rows = spark.sql(f"SELECT batch_id FROM {CHECKPOINT_TABLE}").collect()
completed_batches = {row.batch_id for row in completed_rows}

# --- Staging table (materialized join, partitioned by batch_id) ---
staging_exists = spark.catalog.tableExists(STAGING_TABLE)

if IS_FULL_SYNC and staging_exists and len(completed_batches) > 0:
    # Resuming an interrupted full sync — reuse existing staging data
    total_staged = spark.sql(f"SELECT COUNT(*) FROM {STAGING_TABLE}").collect()[0][0]
    print(f"RESUMING full sync: {len(completed_batches)}/{NUM_BATCHES} batches done")
    print(f"Staging table has {total_staged:,} records")
elif not IS_FULL_SYNC and staging_exists:
    # Incremental: the nightly job's `stage` task (notebooks/elastic/stage_vector_sync_qwen3.sql, SQL warehouse)
    # has just built the staging table — use it as-is. Building it here stalled for hours on the job cluster.
    if RESUME:
        print(f"Incremental RESUME: {len(completed_batches)}/{NUM_BATCHES} batches already done")
    else:
        spark.sql(f"TRUNCATE TABLE {CHECKPOINT_TABLE}")
        completed_batches = set()
    total_staged = spark.sql(f"SELECT COUNT(*) FROM {STAGING_TABLE}").collect()[0][0]
    print(f"Incremental: using staging table built by the stage task: {total_staged:,} records")
else:
    # Fresh start — clear checkpoint and rebuild staging
    spark.sql(f"TRUNCATE TABLE {CHECKPOINT_TABLE}")
    completed_batches = set()

    if IS_FULL_SYNC:
        SQL_QUERY = """
        SELECT
          concat('https://openalex.org/W', e.work_id) as id,
          e.embedding,
          w.publication_year,
          lower(w.type) as type,
          w.open_access.is_oa as is_oa,
          lower(w.language) as language,
          array_compact(transform(w.authorships, a -> a.author.id)) as author_ids,
          array_distinct(array_compact(flatten(
            transform(w.authorships, a -> transform(a.institutions, i -> i.id))
          ))) as institution_ids,
          array_distinct(array_compact(flatten(
            transform(w.authorships, a -> transform(a.institutions, i -> lower(i.country_code)))
          ))) as country_codes,
          w.is_retracted,
          w.primary_location.source.id as source_id,
          w.cited_by_count,
          array_compact(transform(coalesce(w.funders, array()), f -> f.id)) as funder_ids,
          w.fulltext IS NOT NULL as has_fulltext,
          w.has_abstract,
          w.primary_location.license_id as license_id
        FROM openalex.vector_search.work_embeddings_qwen3 e
        JOIN openalex.works.openalex_works w ON e.work_id = CAST(w.id AS STRING)
        """
    else:
        # Incremental: prune openalex_works by an id-only semi-join (broadcast → dynamic file pruning on the
        # clustering column `id`). Joining `e` directly does NOT prune: with the 1024-float `embedding` column the
        # small side is ~8 KB/row (3.7 GB for 460K rows), too big to broadcast, so Spark sort-merges and scans the
        # whole 1.5 TB works table — 6 h on 2 workers with nothing indexed (2026-09-23).
        SQL_QUERY = f"""
        WITH recent AS (
          SELECT * FROM openalex.vector_search.work_embeddings_qwen3
          WHERE embedded_at >= current_timestamp() - INTERVAL {LOOKBACK_DAYS} DAYS
        ),
        w AS (
          SELECT * FROM openalex.works.openalex_works
          WHERE id IN (SELECT CAST(work_id AS BIGINT) FROM recent)
        )
        SELECT
          concat('https://openalex.org/W', e.work_id) as id,
          e.embedding,
          w.publication_year,
          lower(w.type) as type,
          w.open_access.is_oa as is_oa,
          lower(w.language) as language,
          array_compact(transform(w.authorships, a -> a.author.id)) as author_ids,
          array_distinct(array_compact(flatten(
            transform(w.authorships, a -> transform(a.institutions, i -> i.id))
          ))) as institution_ids,
          array_distinct(array_compact(flatten(
            transform(w.authorships, a -> transform(a.institutions, i -> lower(i.country_code)))
          ))) as country_codes,
          w.is_retracted,
          w.primary_location.source.id as source_id,
          w.cited_by_count,
          array_compact(transform(coalesce(w.funders, array()), f -> f.id)) as funder_ids,
          w.fulltext IS NOT NULL as has_fulltext,
          w.has_abstract,
          w.primary_location.license_id as license_id
        FROM recent e
        JOIN w ON w.id = CAST(e.work_id AS BIGINT)
        """

    print(f"Running query and writing staging table ({NUM_BATCHES} batches)...")
    staging_start = time.time()

    df = spark.sql(SQL_QUERY)
    df = df.withColumn("batch_id", F.abs(F.hash("id")) % F.lit(NUM_BATCHES))

    if staging_exists:
        spark.sql(f"DROP TABLE {STAGING_TABLE}")
    df.write.format("delta").partitionBy("batch_id").saveAsTable(STAGING_TABLE)

    total_staged = spark.sql(f"SELECT COUNT(*) FROM {STAGING_TABLE}").collect()[0][0]
    staging_min = (time.time() - staging_start) / 60
    print(f"Staging complete: {total_staged:,} records in {staging_min:.1f}min")

remaining = NUM_BATCHES - len(completed_batches)
print(f"\nReady: {remaining} batches to process")

### Bulk Load Helpers

In [ ]:
log_schema = StructType([
    StructField("partition_id", IntegerType(), True),
    StructField("indexed_count", IntegerType(), True),
    StructField("skipped_count", IntegerType(), True),
    StructField("error_count", IntegerType(), True),
    StructField("errors", ArrayType(StringType()), True)
])


def generate_actions(partition, errors_list):
    """Generate ES bulk index actions for the vector index."""
    for row in partition:
        try:
            embedding = [float(x) for x in row.embedding]

            doc = {
                "id": row.id,
                "vector_embedding": embedding,
                "publication_year": row.publication_year,
                "type": row.type,
                "is_oa": row.is_oa if row.is_oa is not None else False,
                "language": row.language,
                "author_ids": list(row.author_ids) if row.author_ids else [],
                "institution_ids": list(row.institution_ids) if row.institution_ids else [],
                "country_codes": list(row.country_codes) if row.country_codes else [],
                "is_retracted": row.is_retracted if row.is_retracted is not None else False,
                "source_id": row.source_id,
                "cited_by_count": row.cited_by_count or 0,
                "funder_ids": list(row.funder_ids) if row.funder_ids else [],
                "has_fulltext": row.has_fulltext if row.has_fulltext is not None else False,
                "has_abstract": row.has_abstract if row.has_abstract is not None else False,
                "license_id": row.license_id,
            }

            yield {
                "_op_type": "index",
                "_index": ELASTIC_INDEX,
                "_id": row.id,
                "_source": doc
            }
        except Exception as e:
            errors_list.append(f"Parse error for {getattr(row, 'id', '?')}: {str(e)[:200]}")


def send_partition_to_elastic(partition, partition_id):
    """Send a partition of docs to Elasticsearch."""
    client = Elasticsearch(
        hosts=[ELASTIC_URL],
        request_timeout=180,
        max_retries=5,
        retry_on_timeout=True,
        http_compress=True,
    )

    indexed_count = 0
    skipped_count = 0
    errors = []

    try:
        for success, info in helpers.parallel_bulk(
            client,
            generate_actions(partition, errors),
            chunk_size=500,
            thread_count=4,
            queue_size=10,
            raise_on_error=False
        ):
            if success:
                indexed_count += 1
            else:
                error_info = info.get("index", {})
                status = error_info.get("status", 0)
                if status == 409:
                    skipped_count += 1
                else:
                    if len(errors) < 10:
                        errors.append(str(info)[:500])
    except Exception as e:
        errors.append(f"Bulk error: {str(e)[:500]}")
    finally:
        client.close()

    yield {
        "partition_id": partition_id,
        "indexed_count": indexed_count,
        "skipped_count": skipped_count,
        "error_count": len(errors),
        "errors": errors
    }

### Execute Bulk Load

In [ ]:
# Process batches with checkpointing + progress
batches_todo = sorted(b for b in range(NUM_BATCHES) if b not in completed_batches)
if MAX_BATCHES > 0:
    batches_todo = batches_todo[:MAX_BATCHES]
STOP_REASON = None  # set when the pacing (cell 1) ends the run before batches_todo is done


def index_health(client, probe_vectors):
    """(merges running on the index, median `took` ms of 3 direct kNN probes) (oxjob #1433)."""
    import random
    merges = client.indices.stats(index=ELASTIC_INDEX, metric="merge")["_all"]["primaries"]["merges"]["current"]
    took = []
    for _ in range(3):
        v = [x + random.uniform(-0.02, 0.02) for x in random.choice(probe_vectors)]
        try:
            took.append(client.search(index=ELASTIC_INDEX, knn={"field": "vector_embedding", "query_vector": v,
                                                                 "k": 50, "num_candidates": 100},
                                      source=False, size=50)["took"])
        except Exception:
            took.append(99999)
        time.sleep(2)
    return merges, sorted(took)[1]


def wait_until_healthy(client, probe_vectors):
    """True once merges <= MAX_MERGES and kNN <= KNN_GUARD_MS; False after GUARD_WAIT_MINUTES."""
    give_up = time.time() + GUARD_WAIT_MINUTES * 60
    while True:
        merges, knn_ms = index_health(client, probe_vectors)
        ok = merges <= MAX_MERGES and knn_ms <= KNN_GUARD_MS
        print(f"  guard {datetime.utcnow():%H:%M:%S} UTC: merges {merges}, kNN {knn_ms} ms -> {'go' if ok else 'wait'}")
        if ok:
            return True
        if time.time() > give_up:
            return False
        time.sleep(60)


if not batches_todo:
    print("All batches already completed! Nothing to do.")
else:
    # Prior progress from checkpoint
    prior_stats = spark.sql(f"""
        SELECT COALESCE(SUM(indexed_count), 0) as indexed,
               COALESCE(SUM(error_count), 0) as errors
        FROM {CHECKPOINT_TABLE}
    """).collect()[0]
    prior_indexed = prior_stats.indexed

    print(f"=== Starting bulk load to {ELASTIC_INDEX} ===")
    print(f"Batches: {len(batches_todo)} remaining of {NUM_BATCHES}")
    if prior_indexed > 0:
        print(f"Previously indexed: {prior_indexed:,}")
    print()

    start_time = time.time()
    guard_client = None
    if KNN_GUARD_MS > 0:
        guard_client = Elasticsearch(hosts=[ELASTIC_URL], request_timeout=60)
        probe_vectors = [[float(x) for x in r.embedding]
                         for r in spark.read.table(STAGING_TABLE).select("embedding").limit(20).collect()]
    session_indexed = 0
    session_errors = 0

    for i, batch_id in enumerate(batches_todo):
        if MAX_MINUTES > 0 and time.time() - start_time > MAX_MINUTES * 60:
            STOP_REASON = f"max_minutes {MAX_MINUTES} reached"
            break
        if guard_client is not None and not wait_until_healthy(guard_client, probe_vectors):
            STOP_REASON = (f"index not healthy within {GUARD_WAIT_MINUTES} min "
                           f"(max_merges {MAX_MERGES}, knn_guard_ms {KNN_GUARD_MS})")
            break
        batch_start = time.time()

        # Read this batch from staging table (fast — partitioned by batch_id)
        batch_df = spark.read.table(STAGING_TABLE).filter(F.col("batch_id") == batch_id)

        # Send to Elasticsearch
        batch_rdd = batch_df.rdd.mapPartitionsWithIndex(
            lambda idx, part: send_partition_to_elastic(part, idx)
        )
        batch_logs = spark.createDataFrame(batch_rdd, log_schema)

        # Aggregate batch results
        stats = batch_logs.agg(
            F.sum("indexed_count").alias("indexed"),
            F.sum("skipped_count").alias("skipped"),
            F.sum("error_count").alias("errors")
        ).collect()[0]

        batch_indexed = stats.indexed or 0
        batch_skipped = stats.skipped or 0
        batch_errors = stats.errors or 0

        # Checkpoint — survives cluster restarts
        spark.sql(f"""
            INSERT INTO {CHECKPOINT_TABLE}
            VALUES ({batch_id}, {batch_indexed}, {batch_skipped}, {batch_errors}, current_timestamp())
        """)

        # Update session totals
        session_indexed += batch_indexed
        session_errors += batch_errors
        total_indexed = prior_indexed + session_indexed

        # Rate and ETA
        elapsed = time.time() - start_time
        batch_time = time.time() - batch_start
        rate = session_indexed / elapsed if elapsed > 0 else 0
        batches_done = i + 1
        batches_left = len(batches_todo) - batches_done
        avg_batch_time = elapsed / batches_done
        eta_min = (batches_left * avg_batch_time) / 60

        print(
            f"[{batches_done}/{len(batches_todo)}] "
            f"batch {batch_id}: +{batch_indexed:,} ({batch_time:.0f}s) | "
            f"Total: {total_indexed:,} | "
            f"{rate:,.0f}/s | "
            f"ETA: {eta_min:.0f}min"
        )

        # Show sample errors if any
        if batch_errors > 0:
            error_rows = batch_logs.filter(F.size("errors") > 0).select("errors").limit(3).collect()
            for row in error_rows:
                for err in row.errors[:1]:
                    print(f"  ERROR: {err[:300]}")

    if guard_client is not None:
        guard_client.close()
    if STOP_REASON:
        print(f"\nSTOPPED EARLY: {STOP_REASON}. resume=true continues from the checkpoint.")
    total_elapsed = (time.time() - start_time) / 60
    print(f"\n=== Session Complete ===")
    print(f"Session: {session_indexed:,} indexed, {session_errors:,} errors in {total_elapsed:.1f}min")
    print(f"All sessions: {prior_indexed + session_indexed:,} total indexed")

In [ ]:
# Deletes (oxjob #1433): works that no longer exist, listed by the stage task. A 404 means already gone.
DELETED_OK = None  # None = no deletes step ran; else whether every delete went through
ALL_BATCHES_DONE = spark.sql(f"SELECT COUNT(DISTINCT batch_id) FROM {CHECKPOINT_TABLE}").collect()[0][0] >= NUM_BATCHES
if not ALL_BATCHES_DONE:
    print("Batches still outstanding (paced run, oxjob #1433): the deletes wait for the run that finishes them")
elif DELETES_TABLE.lower() != "none" and spark.catalog.tableExists(DELETES_TABLE):
    n_deletes = spark.table(DELETES_TABLE).count()
    print(f"Deletes listed in {DELETES_TABLE}: {n_deletes:,}")
    if n_deletes > MAX_DELETES:
        raise Exception(
            f"{n_deletes:,} deletes > max_deletes {MAX_DELETES:,}: refusing. A read of openalex_works during a rebuild "
            f"looks like this. Check the list; if it is real, re-run with max_deletes raised."
        )

    def delete_partition(partition):
        client = Elasticsearch(hosts=[ELASTIC_URL], request_timeout=180, max_retries=5, retry_on_timeout=True)
        deleted = missing = errors = 0
        try:
            actions = ({"_op_type": "delete", "_index": ELASTIC_INDEX, "_id": row.id} for row in partition)
            for ok, info in helpers.parallel_bulk(client, actions, chunk_size=1000, thread_count=4,
                                                  queue_size=10, raise_on_error=False):
                if ok:
                    deleted += 1
                elif info.get("delete", {}).get("status") == 404:
                    missing += 1
                else:
                    errors += 1
        except Exception:
            errors += 1
        finally:
            client.close()
        yield (deleted, missing, errors)

    totals = (0, 0, 0)
    if n_deletes:
        totals = (spark.table(DELETES_TABLE).select("id").repartition(max(8, n_deletes // 50000))
                  .rdd.mapPartitions(delete_partition)
                  .reduce(lambda a, b: tuple(x + y for x, y in zip(a, b))))
    print(f"Deleted {totals[0]:,}, already gone {totals[1]:,}, errors {totals[2]:,}")
    DELETED_OK = totals[2] == 0
else:
    print(f"No deletes table ({DELETES_TABLE}); skipping deletes")


In [ ]:
# Final summary from checkpoint table
summary = spark.sql(f"""
    SELECT
        COUNT(*) as batches_done,
        COALESCE(SUM(indexed_count), 0) as total_indexed,
        COALESCE(SUM(skipped_count), 0) as total_skipped,
        COALESCE(SUM(error_count), 0) as total_errors,
        MIN(completed_at) as first_batch_at,
        MAX(completed_at) as last_batch_at
    FROM {CHECKPOINT_TABLE}
""").collect()[0]

print(f"=== Sync Summary (from checkpoint) ===")
print(f"Batches completed: {summary.batches_done}/{NUM_BATCHES}")
print(f"Total indexed: {summary.total_indexed:,}")
print(f"Total skipped: {summary.total_skipped:,}")
print(f"Total errors:  {summary.total_errors:,}")
if summary.first_batch_at and summary.last_batch_at:
    print(f"Time span: {summary.first_batch_at} → {summary.last_batch_at}")

if summary.batches_done < NUM_BATCHES:
    missing = NUM_BATCHES - summary.batches_done
    print(f"\nWARNING: {missing} batches not yet completed. Re-run cell 8 to resume.")

In [ ]:
# Post-sync: refresh, set refresh_interval, verify doc count, clean up
# NOTE: Do NOT force merge here — run graduated merge manually (see PLAN.md)
# NOTE: Do NOT set replicas=1 — keep at 0 permanently (9.3TB, rebuildable in ~4h)
try:
    client = Elasticsearch(hosts=[ELASTIC_URL], request_timeout=180)

    if client.indices.exists(index=ELASTIC_INDEX):
        # Refresh
        client.indices.refresh(index=ELASTIC_INDEX)
        print(f"Refreshed index {ELASTIC_INDEX}")

        # Doc count
        total_docs = client.count(index=ELASTIC_INDEX)['count']
        print(f"Total documents: {total_docs:,}")

        # Set refresh interval (but keep replicas=0)
        if IS_FULL_SYNC:
            client.indices.put_settings(index=ELASTIC_INDEX, body={
                "index": {
                    "refresh_interval": "30s"
                }
            })
            print(f"Set refresh_interval=30s on {ELASTIC_INDEX} (replicas stay at 0)")
    else:
        print(f"Index {ELASTIC_INDEX} does not exist")
finally:
    client.close()

# Mirror (oxjob #1433): after a clean sync, record the filter-field hash of every staged work so the next stage task
# re-sends only works whose fields changed since. Skipped on any error: the next stage task then re-sends them.
if (ALL_BATCHES_DONE and MIRROR_TABLE.lower() != "none" and spark.catalog.tableExists(MIRROR_TABLE)
        and "filter_hash" in spark.table(STAGING_TABLE).columns):
    if summary.batches_done == NUM_BATCHES and summary.total_errors == 0 and DELETED_OK is not False:
        spark.sql(f"""
            MERGE INTO {MIRROR_TABLE} m
            USING (SELECT DISTINCT id, filter_hash FROM {STAGING_TABLE}) s ON m.id = s.id
            WHEN MATCHED THEN UPDATE SET m.filter_hash = s.filter_hash, m.sent_at = current_timestamp()
            WHEN NOT MATCHED THEN INSERT (id, filter_hash, sent_at) VALUES (s.id, s.filter_hash, current_timestamp())
        """)
        if DELETED_OK:
            spark.sql(f"DELETE FROM {MIRROR_TABLE} WHERE id IN (SELECT id FROM {DELETES_TABLE})")
        print(f"Mirror {MIRROR_TABLE} updated")
    else:
        print(f"Mirror NOT updated (batches {summary.batches_done}/{NUM_BATCHES}, errors {summary.total_errors}, "
              f"deletes ok: {DELETED_OK}); the next stage task re-sends these works")

# Clean up staging, checkpoint and deletes tables (cleanup=false keeps them, e.g. to check a one-off run)
if CLEANUP and not ALL_BATCHES_DONE:
    print("\nBatches still outstanding: staging, checkpoint and deletes tables kept for resume=true")
elif CLEANUP:
    print(f"\nCleaning up temp tables...")
    spark.sql(f"DROP TABLE IF EXISTS {STAGING_TABLE}")
    spark.sql(f"DROP TABLE IF EXISTS {CHECKPOINT_TABLE}")
    if DELETES_TABLE.lower() != "none":
        spark.sql(f"DROP TABLE IF EXISTS {DELETES_TABLE}")
    print("Staging, checkpoint and deletes tables dropped.")
else:
    print("cleanup=false: staging, checkpoint and deletes tables kept")

In [ ]:
# Test kNN search with a sample query
print("Testing kNN search...")

try:
    client = Elasticsearch(hosts=[ELASTIC_URL], request_timeout=180)

    # Get a sample embedding to use as query
    sample = spark.sql("""
        SELECT work_id, embedding
        FROM openalex.vector_search.work_embeddings_qwen3
        LIMIT 1
    """).collect()[0]

    query_vector = [float(x) for x in sample.embedding]

    # Run kNN search on the vector index
    result = client.search(
        index=ELASTIC_INDEX,
        body={
            "knn": {
                "field": "vector_embedding",
                "query_vector": query_vector,
                "k": 5,
                "num_candidates": 50
            },
            "_source": ["id", "publication_year", "type", "cited_by_count"]
        },
        size=5
    )

    print(f"Query work_id: {sample.work_id}")
    print(f"\nTop 5 similar works:")
    for hit in result['hits']['hits']:
        src = hit['_source']
        print(f"  {hit['_score']:.4f}: {src.get('id', 'N/A')} ({src.get('type', '?')}, {src.get('publication_year', '?')}, cited: {src.get('cited_by_count', 0)})")

    # Test with a filter
    filtered_result = client.search(
        index=ELASTIC_INDEX,
        body={
            "knn": {
                "field": "vector_embedding",
                "query_vector": query_vector,
                "k": 5,
                "num_candidates": 50,
                "filter": {
                    "bool": {
                        "must": [
                            {"term": {"is_oa": True}},
                            {"range": {"publication_year": {"gte": 2020}}}
                        ]
                    }
                }
            },
            "_source": ["id", "publication_year", "is_oa"]
        },
        size=5
    )

    print(f"\nFiltered (is_oa=true, year>=2020):")
    for hit in filtered_result['hits']['hits']:
        src = hit['_source']
        print(f"  {hit['_score']:.4f}: {src.get('id', 'N/A')} ({src.get('publication_year', '?')}, is_oa={src.get('is_oa', '?')})")

finally:
    client.close()